In [13]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import IncrementalPCA
from sklearn.decomposition import PCA
from pathlib import Path

In [18]:
all_feat_dir = Path('/media/yiting/NewVolume/Analysis/shape_analysis/shape_rdms/all_features')
red_feat_dir = Path('/media/yiting/NewVolume/Analysis/shape_analysis/shape_rdms')
feat_fname = 'alexnet_low_features_concatenated_rgb_correct_all.npy'
all_feat_path = all_feat_dir / feat_fname
red_feat_path = red_feat_dir / feat_fname
features = np.load(all_feat_path)

In [19]:
# 1. Force float32 to save 50% memory
# If X_low is already in memory, convert it; if loading from disk, use mmap
X = features.astype(np.float32)

# 2. Set n_components to the maximum possible (N_samples - 1)
# You cannot extract 4096 components from 414 shapes.
n_samples = X.shape[0]
n_components = n_samples - 1 

print(f"Reducing {X.shape[1]} features to {n_components} components...")

# 3A. Fit and Transform
# Use a smaller batch size to keep memory usage low
# ipca = IncrementalPCA(n_components=n_components, batch_size=50)
# X_low_reduced = ipca.fit_transform(X_low)

# 3B. Randomized solver is highly optimized for matrices where features >> samples
pca = PCA(n_components=n_components, svd_solver='randomized', random_state=42)
X_reduced = pca.fit_transform(X)

# 4. Calculate Variance
# exp_var_ratio = ipca.explained_variance_ratio_
exp_var_ratio = pca.explained_variance_ratio_
cum_exp_var = np.cumsum(exp_var_ratio)

print(f"Total variance explained by {n_components} components: {cum_exp_var[-1]*100:.2f}%")

np.save(red_feat_path, X_reduced)

Reducing 1161600 features to 413 components...
Total variance explained by 413 components: 100.00%
